In [112]:
import pandas as pd
import numpy as np
import xgboost as xgb
from lightgbm import LGBMRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Creazione e addestramento dei modelli

In [113]:
def load_and_split_data(train_path, test_path, target_col='target_assignments'):
    """Carica i dati encodati, separa feature e target, e preserva gli ID."""
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    
    test_ids = df_test['operator_id']
    
    X_train = df_train.drop(columns=[target_col, 'operator_id'])
    y_train = df_train[target_col]
    
    X_test = df_test.drop(columns=[target_col, 'operator_id'])
    y_test = df_test[target_col]
    
    print(f"Dataset caricati: Train ({X_train.shape[0]} righe), Test ({X_test.shape[0]} righe)")
    return X_train, y_train, X_test, y_test, test_ids

X_train, y_train, X_test, y_test, test_ids = load_and_split_data('data/train_dataset.csv', 'data/test_dataset.csv')

Dataset caricati: Train (1738 righe), Test (435 righe)


In [114]:
def prepare_asp_facts(X_test, y_test, q10, q50, q90):
    """
    Combina le predizioni con i dati di contesto e calcola lo score di incertezza.
    """
    results_df = X_test.copy()
    
    # La predizione da suggerire all'ASP arrotondata all'intero più vicino
    results_df['predicted_assignments'] = np.round(q50).astype(int)
    
    # Valore reale
    results_df['actual_assignments'] = y_test.values
    
    # Calcolo dell'Incertezza
    # Più il valore è alto, maggiore è l'incertezza del modello.
    results_df['uncertainty_score'] = q90 - q10
    
    # Estrazione del burdenScore
    burden_col = [col for col in X_test.columns if 'burdenScore' in col][0]
    
    print("\nAnteprima dei dati pronti per essere convertiti in fatti ASP:")
    display_cols = ['predicted_assignments', 'uncertainty_score', burden_col, 'density_ratio']
    display(results_df[display_cols].head())
    
    return results_df

In [115]:
def train_xgboost_quantiles(X_train, y_train, X_test):
    preds = {}
    trained_models = {}
    for q, alpha in [('q10', 0.10), ('q50', 0.50), ('q90', 0.90)]:
        model = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=alpha, 
                                 n_estimators=100, learning_rate=0.1, max_depth=4, random_state=42)
        model.fit(X_train, y_train)
        preds[q] = model.predict(X_test)
        trained_models[q] = model
    return trained_models, preds['q10'], preds['q50'], preds['q90']

def train_lightgbm_quantiles(X_train, y_train, X_test):
    preds = {}
    trained_models = {}
    for q, alpha in [('q10', 0.10), ('q50', 0.50), ('q90', 0.90)]:
        model = LGBMRegressor(objective='quantile', alpha=alpha, 
                              n_estimators=100, random_state=42, verbose=-1)
        model.fit(X_train, y_train)
        preds[q] = model.predict(X_test)
        trained_models[q] = model
    return trained_models, preds['q10'], preds['q50'], preds['q90']

def train_sklearn_gb_quantiles(X_train, y_train, X_test):
    preds = {}
    trained_models = {}
    for q, alpha in [('q10', 0.10), ('q50', 0.50), ('q90', 0.90)]:
        model = GradientBoostingRegressor(loss='quantile', alpha=alpha, 
                                          n_estimators=100, random_state=42)
        model.fit(X_train, y_train)
        preds[q] = model.predict(X_test)
        trained_models[q] = model
    return trained_models, preds['q10'], preds['q50'], preds['q90']

In [ ]:
def evaluate_model(model_name, y_test, q50, q10, q90):
    mae = mean_absolute_error(y_test, q50)
    rmse = np.sqrt(mean_squared_error(y_test, q50))
    r2 = r2_score(y_test, q50)
    coverage = np.mean((y_test >= q10) & (y_test <= q90))

    print(f"{model_name:<12} | MAE: {mae:.3f} | RMSE: {rmse:.3f} | R^2: {r2:.3f} | Copertura: {coverage*100:.1f}%")
    return mae

print("--- Addestramento e Valutazione Modelli ---")
models = {
    "XGBoost": train_xgboost_quantiles,
    "LightGBM": train_lightgbm_quantiles,
    "Sklearn GB": train_sklearn_gb_quantiles
}

best_mae = float('inf')
best_model_name = ""
best_predictions = None
best_trained_models = None

for name, train_func in models.items():
    trained_models, q10, q50, q90 = train_func(X_train, y_train, X_test)
    mae = evaluate_model(name, y_test, q50, q10, q90)
    
    if mae < best_mae:
        best_mae = mae
        best_model_name = name
        best_predictions = (q10, q50, q90)
        best_trained_models = trained_models

--- Addestramento e Valutazione Modelli ---
XGBoost      | MAE: 0.435 | RMSE: 0.646 | R^2: 0.740 | Copertura: 94.0%
LightGBM     | MAE: 0.400 | RMSE: 0.627 | R^2: 0.754 | Copertura: 87.4%
Sklearn GB   | MAE: 0.398 | RMSE: 0.614 | R^2: 0.764 | Copertura: 94.5%


In [117]:
def generate_clingo_facts(X_test, y_test, q10, q50, q90, original_ids, output_filename):
    df_facts = pd.DataFrame({
        'operator_id': original_ids,
        'predicted_assignments': np.round(q50).astype(int),
        'actual_assignments': y_test.values,
        'uncertainty_score': q90 - q10,
        'op_burdenScore': X_test['op_burdenScore'].values 
    })
    
    df_facts['ConfLevel'] = pd.qcut(df_facts['uncertainty_score'].rank(method='first'), q=4, labels=[1, 2, 3, 4]).astype(int)
    
    asp_lines = [
        "% --- FATTI GENERATI DAL MACHINE LEARNING ---",
        f"% Modello utilizzato: {best_model_name}",
        "% Formato: ml_capacity(OP_ID, PRED_N, CONF_LEVEL).",
        "% Formato: operator_burden(OP_ID, BURDEN_SCORE).",
    ]
    
    for _, row in df_facts.iterrows():
        op_id = str(row['operator_id']).replace('.0', '')
        if op_id != '-1':
            asp_lines.append(f"ml_capacity({op_id}, {int(row['predicted_assignments'])}, {int(row['ConfLevel'])}).")
            asp_lines.append(f"operator_burden({op_id}, {int(row['op_burdenScore'])}).")
        
    with open(output_filename, 'w') as f:
        f.write("\n".join(asp_lines))
        
    print(f"File '{output_filename}' generato con {len(df_facts[df_facts['operator_id'] != -1])} record validi.")
    return df_facts

In [ ]:
import json

def generate_physical_instance(json_path, target_date, output_filename):
    with open(json_path, 'r', encoding='utf-8') as f:
        docs = json.load(f)
        docs = [docs] if isinstance(docs, dict) else docs
        
    asp_lines = [f"% --- FATTI FISICI DELLA GIORNATA: {target_date} ---"]
    
    for doc in docs:
        planning_date = doc.get('planningDate', {}).get('$date', '')
        if target_date not in planning_date:
            continue
            
        board = doc.get('board', [])
        unassigned = doc.get('unassignedPatients', [])
        agenda = doc.get('agenda', [])
        
        # 1. ESTRAZIONE OPERATORI
        asp_lines.append("% --- OPERATORI ---")
        for op in board:
            op_id = op.get('id')
            eff_time = op.get('effectiveTime', 0) // 10
            is_pt = 1 if 'part-time' in str(op.get('jobKind', '')).lower() else 0
            max_pats = (eff_time // 3) if eff_time > 0 else 10 
            
            quals = op.get('qualifications', [])
            limit_n = max_pats if 'N' in quals else 0
            limit_o = max_pats if 'O' in quals else 0
            limit_cp = max_pats if 'CP' in quals else 0
            limit_cn = max_pats if 'CN' in quals else 0
            limit_mac = max_pats if 'MAC' in quals else 0
            
            asp_lines.append(
                f"operator({op_id}, {eff_time}, {is_pt}, {max_pats}, "
                f"{limit_n}, {limit_n}, {limit_n}, {limit_n}, "
                f"{limit_o}, {limit_o}, {limit_o}, {limit_o}, "
                f"{limit_cp}, {limit_cn}, {limit_mac})."
            )
        asp_lines.append("operator(-1, 100, 0, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100, 100).\n")

        # 2. ESTRAZIONE PAZIENTI E PREFERENZE
        asp_lines.append("% --- PAZIENTI E PREFERENZE ---")
        all_patients = unassigned.copy()
        for op in board:
            all_patients.extend(op.get('patients', []))
            
        seen_pats = set()
        for pat in all_patients:
            pat_id = pat.get('id')
            if pat_id in seen_pats:
                continue
            seen_pats.add(pat_id)
            
            p_type_str = pat.get('type', '')
            type_map = {'N': 1, 'O': 2, 'A': 3, 'CP': 4, 'CN': 5, 'MAC': 6}
            p_type = type_map.get(p_type_str, 1)
            
            is_paying = 2 if 'ssn' in str(pat.get('statusStr', '')).lower() else 1
            needs_aid = 0 if str(pat.get('aidNeeds')).lower() in ['none', 'null', '', 'false'] else 1
            min_len = pat.get('overallMinLength', 60) // 10
            
            asp_lines.append(f"patient({pat_id}, {p_type}, {is_paying}, {needs_aid}, {min_len}).")
            
            # --- ESTRAZIONE PREFERENZE ---
            asp_lines.append(f"pref(-1, {pat_id}, 10, 1).")
            
            pref_ops = pat.get('preferredOps', [])
            for weight, ops_group in enumerate(pref_ops):
                if isinstance(ops_group, list):
                    for pref_op_id in ops_group:
                        asp_lines.append(f"pref({pref_op_id}, {pat_id}, {weight}, 1).")
        asp_lines.append("\n")

        # 3. ESTRAZIONE SESSIONI
        asp_lines.append("% --- SESSIONI CLINICHE ---")
        seen_sessions = set()
        for item in agenda:
            sess = item.get('session')
            pat = item.get('patient')
            
            if not sess or not pat:
                continue
                
            sess_id = sess.get('id')
            if sess_id in seen_sessions:
                continue
            seen_sessions.add(sess_id)
            
            pat_id = pat.get('id')
            min_len = sess.get('minLength', 60) // 10
            
            loc_str = str(sess.get('location', '1')).strip()
            loc = loc_str if loc_str.isdigit() else 1 
            
            asp_lines.append(f"session({sess_id}, {pat_id}, {min_len}, {loc}).")

    with open(output_filename, 'w') as f:
        f.write("\n".join(asp_lines))

In [ ]:
import pandas as pd
import numpy as np
import import_ipynb
from pre_processing_new import extract_from_json, aggregate_to_operator_day

def generate_predictions_for_date(json_path, target_date, train_columns, trained_models, output_filename):
    df_raw = extract_from_json([json_path])
    df_agg = aggregate_to_operator_day(df_raw)
    
    df_today = df_agg[df_agg['planning_date'].astype(str).str.contains(target_date)].copy()
    
    if df_today.empty:
        print("Nessun dato trovato per questa data.")
        return
        
    today_ids = df_today['operator_id']
    X_today = df_today.drop(columns=['operator_id', 'planning_date', 'target_assignments'], errors='ignore')
    
    cat_cols = X_today.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    X_today = pd.get_dummies(X_today, columns=cat_cols, drop_first=True, dtype=int)
    
    X_today = X_today.reindex(columns=train_columns, fill_value=0)
    
    q10 = trained_models['q10'].predict(X_today)
    q50 = trained_models['q50'].predict(X_today)
    q90 = trained_models['q90'].predict(X_today)
    
    y_dummy = np.zeros(len(today_ids))
    
    generate_clingo_facts(
        X_test=df_today,
        y_test=pd.Series(y_dummy), 
        q10=q10,
        q50=q50,
        q90=q90,
        original_ids=today_ids, 
        output_filename=output_filename
    )

In [ ]:
import clingo

def run_neuro_symbolic_solver(rules_file, facts_file, ml_file, timeout_seconds=30.0):    
    ctl = clingo.Control(["--opt-strategy=usc,k,0,4", "--opt-usc-shrink=bin"])
    
    ctl.load(rules_file)
    ctl.load(facts_file)
    if ml_file:
        ctl.load(ml_file)
    
    ctl.ground([("base", [])])
    
    best_cost = None
    assignments = []
    unassigned_count = 0
    
    def on_model(model):
        nonlocal best_cost, assignments, unassigned_count
        best_cost = model.cost
        assignments = [str(sym) for sym in model.symbols(shown=True)]
        unassigned_count = sum(1 for a in assignments if "assignment(-1" in a)
        print(f"Modello trovato con costo: {model.cost}")
    
    with ctl.solve(on_model=on_model, async_=True) as handle:
        finished = handle.wait(timeout_seconds)
        
        if not finished:
            print(f"Time-limit di {timeout_seconds}s raggiunto")
            handle.cancel()
            
        result = handle.get()
    
    print("\n--- RISULTATO OTTIMIZZAZIONE ---")
    print(f"Stato: {result}")
    print(f"Costo di Ottimizzazione Finale: {best_cost}")
    print(f"Pazienti Non Assegnati (Operatore -1): {unassigned_count}")
    print(f"Totale Assegnazioni Create: {len(assignments)}\n\n\n")
    
    return assignments, best_cost, unassigned_count

In [121]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import os

def run_monthly_experiment(json_path, start_date, end_date, train_columns, trained_models, timeout=30.0):
    dates = pd.date_range(start=start_date, end=end_date)
    results = []
    
    for date_obj in dates:
        date_str = date_obj.strftime('%Y-%m-%d')
        print(f"TEST GIORNO: {date_str}")
        
        try:
            generate_physical_instance(json_path, date_str, "encodings/facts/day_facts.lp")
        except Exception as e:
            print(f"Saltato: Nessun dato fisico trovato per il {date_str}.")
            continue
            
        with open("encodings/facts/day_facts.lp", 'r') as f:
            if len(f.readlines()) < 10:
                print(f"Saltato: Ospedale vuoto il {date_str}.")
                continue
        
        try:
            generate_predictions_for_date(json_path, date_str, train_columns, trained_models, "encodings/facts/ml_facts.lp")
        except Exception as e:
            print(f"Saltato: Impossibile generare predizioni ML per il {date_str}.")
            continue
            
        ass_base, cost_base, unass_base = run_neuro_symbolic_solver(
            rules_file="encodings/baseline_encoding.lp",
            facts_file="encodings/facts/day_facts.lp",
            ml_file=None,
            timeout_seconds=timeout
        )
        
        ass_ns, cost_ns, unass_ns = run_neuro_symbolic_solver(
            rules_file="encodings/ml_encoding.lp",
            facts_file="encodings/facts/day_facts.lp",
            ml_file="encodings/facts/ml_facts.lp",
            timeout_seconds=timeout
        )
        
        cost_base_list = cost_base if cost_base is not None else [0, 0, 0]
        cost_ns_list = cost_ns if cost_ns is not None else [0, 0, 0]
        
        results.append({
            'Data': date_str,
            'Pazienti_A_Terra_Baseline': unass_base,
            'Pazienti_A_Terra_NS': unass_ns,
            'Assegnazioni_Totali_Baseline': len(ass_base) - unass_base,
            'Assegnazioni_Totali_NS': len(ass_ns) - unass_ns,
            'Costo_W2_Baseline': cost_base_list[1] if len(cost_base_list) > 1 else 0,
            'Costo_W2_NS': cost_ns_list[1] if len(cost_ns_list) > 1 else 0
        })

    df_results = pd.DataFrame(results)
    return df_results

data_url = 'data/anon_boardHistory_Org6_2023-01-01_2023-01-31.json'
df_esperimento = run_monthly_experiment(
    json_path=data_url, 
    start_date='2023-01-01', 
    end_date='2023-01-31', 
    train_columns=X_train.columns, 
    trained_models=best_trained_models,
    timeout=20.0
)

display(df_esperimento)

TEST GIORNO: 2023-01-01
Saltato: Ospedale vuoto il 2023-01-01.
TEST GIORNO: 2023-01-02
Estrazione completata: 2187 sessioni analizzate.
Aggregazione completata: Dataset Operatore-Giorno generato con 421 record.
File 'encodings/facts/ml_facts.lp' generato con 19 record validi.
Modello trovato con costo: [10, 1, 0]

--- RISULTATO OTTIMIZZAZIONE ---
Stato: SAT
Costo di Ottimizzazione Finale: [10, 1, 0]
Pazienti Non Assegnati (Operatore -1): 1
Totale Assegnazioni Create: 89



Modello trovato con costo: [140, 2, 18]
Modello trovato con costo: [130, 1, 18]

--- RISULTATO OTTIMIZZAZIONE ---
Stato: SAT
Costo di Ottimizzazione Finale: [130, 1, 18]
Pazienti Non Assegnati (Operatore -1): 1
Totale Assegnazioni Create: 89



TEST GIORNO: 2023-01-03
Estrazione completata: 2187 sessioni analizzate.
Aggregazione completata: Dataset Operatore-Giorno generato con 421 record.
File 'encodings/facts/ml_facts.lp' generato con 16 record validi.
Modello trovato con costo: [190, 19, 0]
Modello trovato con cos

,Data,Pazienti_A_Terra_Baseline,Pazienti_A_Terra_NS,Assegnazioni_Totali_Baseline,Assegnazioni_Totali_NS,Costo_W2_Baseline,Costo_W2_NS
0,2023-01-02,1,1,88,88,1,1
1,2023-01-03,1,1,60,60,1,1
2,2023-01-04,0,0,96,96,0,0
3,2023-01-05,0,0,99,99,0,0
4,2023-01-09,0,0,100,100,0,0
5,2023-01-10,0,0,102,102,0,0
6,2023-01-11,0,0,105,105,0,0
7,2023-01-12,0,0,112,112,0,98
8,2023-01-13,0,0,114,114,0,0
9,2023-01-16,0,0,112,112,0,0
